# B3：SUMO CACC 编队协同巡航 —— 真闭环仿真

## 实验目标
- 理解 CACC 编队跟驰原理（固定时距策略）
- 实现 `CACCController` 并验证弦稳定性
- 掌握编队间距误差的量化评测

## 通过判据
```python
assert max_spacing_error <= 1.0  # 60s 编队无解体
```

> 注：完整 SUMO TraCI 需路网文件，本实验用运动学模型做真闭环，
> 控制逻辑与 TraCI 版本一致（见挑战任务）。


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import KinematicBicycleModel
import math
print('✓ 导入成功')


## 1. CACC 控制器
继承 `BaseController`，实现固定时距跟驰。

In [ ]:
class CACCController(BaseController):
    """协同自适应巡航：跟随前车，保持固定时距"""
    def __init__(self, config=None):
        super().__init__(config)
        self.headway = self.config.get('headway', 0.8)
        self.kp, self.kd = 1.2, 0.8
    def step(self, observation):
        ego, targets = observation['ego'], observation.get('targets', [])
        if not targets:
            return {'target_a': 0.5, 'steer_angle': 0.0}
        lead = min(targets, key=lambda t: t['x'])
        gap_err = (lead['x'] - ego['x'] - 5.0) - ego['v'] * self.headway
        v_err = lead['v'] - ego['v']
        a = self.kp * gap_err + self.kd * v_err
        return {'target_a': max(-3.0, min(2.0, a)), 'steer_angle': 0.0}

print('✓ CACC 控制器定义完成')


## 2. 真闭环编队仿真
1 领航车 + 3 跟随车，20Hz 真闭环。

In [ ]:
# 编队：leader(0) + follower1(1) + follower2(2) + follower3(3)
vehicles = [KinematicBicycleModel() for _ in range(4)]
# 初始位置：每车间隔 15m，速度 15m/s
for i, v in enumerate(vehicles):
    v.reset(x=100 - i*17, y=0, yaw=0, v=15.0)

controllers = [CACCController(config={'headway': 0.8}) for _ in range(3)]
for c in controllers: c.reset()

dt = 0.05
history = []  # (t, [x0,x1,x2,x3], [v0,v1,v2,v3])
max_spacing_error = 0.0

for step in range(1200):  # 60s
    t = step * dt
    # Leader: 5s 后减速到 12m/s（模拟扰动），20s 后恢复
    if 5 < t < 20:
        leader_a = -0.5
    elif 20 <= t < 25:
        leader_a = 0.6
    else:
        leader_a = 0.0
    vehicles[0].step(leader_a, 0, dt)
    
    # Followers: CACC 闭环
    for i in range(1, 4):
        ego, lead = vehicles[i], vehicles[i-1]
        obs = {'ego': {'x': ego.x, 'y': 0, 'yaw': 0, 'v': ego.v, 'a': 0},
               'targets': [{'id': i-1, 'x': lead.x, 'y': 0, 'v': lead.v}],
               'sim_time': t, 'dt': dt}
        act = controllers[i-1].step(obs)
        ego.step(act['target_a'], 0, dt)
        # 间距误差
        desired = ego.v * 0.8 + 5.0
        actual = lead.x - ego.x
        max_spacing_error = max(max_spacing_error, abs(actual - desired))
    
    history.append((t, [v.x for v in vehicles], [v.v for v in vehicles]))

print(f'✓ 编队仿真完成：{len(history)} 步')
print(f'  最大间距误差 = {max_spacing_error:.2f}m')


## 3. 评测断言

In [ ]:
# 通过判据：60s 内最大间距误差 ≤ 1.0m
assert max_spacing_error <= 1.0, f'间距误差 {max_spacing_error:.2f}m > 1.0m，编队不稳定！'
print('✅ 评测通过！编队弦稳定')


## 4. 编队可视化

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

times = [t for t, xs, vs in history]
speeds = np.array([vs for t, xs, vs in history])
positions = np.array([xs for t, xs, vs in history])

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# 速度曲线
for i in range(4):
    label = 'Leader' if i == 0 else f'Follower {i}'
    axes[0].plot(times, speeds[:, i], label=label)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Speed (m/s)')
axes[0].set_title('Platoon Speed - String Stability Test')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 间距曲线
for i in range(1, 4):
    gap = positions[:, i-1] - positions[:, i]
    axes[1].plot(times, gap, label=f'Gap {i-1}-{i}')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Gap (m)')
axes[1].set_title('Inter-vehicle Gaps')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. 🏆 挑战任务
1. **参数挑战**：把 `headway` 改为 0.6 和 1.0，观察间距误差变化。哪个更稳定？
2. **TraCI 挑战**：用 `netgenerate` 生成直道路网，把上面的控制逻辑移植到 TraCI（`traci.vehicle.setSpeed`），对比结果。
3. **扰动挑战**：让 Leader 做正弦速度扰动（±2m/s），看编队能否保持稳定。
